# Phase 6: Vector Database Storage

**Project:** Hospital Patient Helpdesk Chatbot  
**Purpose:** Persist Phase 5 embeddings in a queryable vector index with text, provenance, and metadata filters.

This notebook imports `04_embeddings/06_store_vector_index.py`; the Python module remains the single source of truth.

## Input and output contract

**Inputs:** `01_data/processed/05_embeddings.json` and `01_data/processed/04_enriched_chunks.json`.

**Index output:** `05_vector_store/chroma_db/06_vector_index.sqlite3`.

**Data outputs:** `06_vector_index_manifest.json`, `06_vector_index_report.json`, `06_vector_index_audit.csv`, and `06_failed_index_records.json`.

**Plots:** `plots/06_vectors_by_department.png` and `plots/06_vectors_by_source_type.png`.

## Backend choice

The active environment does not include ChromaDB or FAISS, so this phase uses a portable SQLite exact-cosine backend. It stores float32 vectors, exact text, checksums, and retrieval metadata, and supports department/category filtering. Its interface is intentionally backend-neutral so ChromaDB can replace it later without changing upstream artifacts.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Locate the project from the workspace, project, or notebooks folder."""
    start = start.resolve()
    candidates = []
    for directory in (start, *start.parents):
        candidates.extend((directory, directory / "hospital_patient_helpdesk_chatbot"))
    for candidate in candidates:
        module = candidate / "04_embeddings" / "06_store_vector_index.py"
        embeddings_file = candidate / "01_data" / "processed" / "05_embeddings.json"
        chunks_file = candidate / "01_data" / "processed" / "04_enriched_chunks.json"
        if module.is_file() and embeddings_file.is_file() and chunks_file.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the hospital_patient_helpdesk_chatbot project. "
        "Open this notebook from the project workspace or keep it under 13_notebooks."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
EMBEDDINGS_PATH = PROJECT_ROOT / "01_data" / "processed" / "05_embeddings.json"
CHUNKS_PATH = PROJECT_ROOT / "01_data" / "processed" / "04_enriched_chunks.json"
OUTPUT_DIR = PROJECT_ROOT / "01_data" / "processed"
INDEX_DIR = PROJECT_ROOT / "05_vector_store" / "chroma_db"
MODULE_PATH = PROJECT_ROOT / "04_embeddings" / "06_store_vector_index.py"
spec = importlib.util.spec_from_file_location("store_vector_index", MODULE_PATH)
vector_store = importlib.util.module_from_spec(spec)
assert spec and spec.loader
sys.modules[spec.name] = vector_store
spec.loader.exec_module(vector_store)
print(f"Project root: {PROJECT_ROOT}")
print(f"Backend: {vector_store.BACKEND_NAME}")

## 1. Inspect and validate inputs

Phase 5 provides vectors and checksums. Phase 4 provides exact source text and full retrieval metadata. Validation requires matching IDs, one model, one dimension, and matching text checksums.

In [ ]:
embedding_records = vector_store.load_json_list(EMBEDDINGS_PATH, "Embedding input")
enriched_chunks = vector_store.load_json_list(CHUNKS_PATH, "Enriched chunk input")
dimension, model = vector_store.validate_inputs(embedding_records, enriched_chunks)
print(f"Embedding records: {len(embedding_records)}")
print(f"Dimension: {dimension}")
print(f"Model: {model}")

## 2. Build and validate the persistent index

The module writes a complete temporary database, validates SQLite integrity, closes all handles, and atomically promotes the file. This prevents a partial index from replacing a known-good index.

In [ ]:
result = vector_store.run_index_storage(EMBEDDINGS_PATH, CHUNKS_PATH, INDEX_DIR, OUTPUT_DIR)
vector_store.print_result(result)
vector_store.validate_index(result.index_path, len(embedding_records), dimension)
print("Persistent index validation passed.")

## 3. Run exact cosine queries

The first example uses an indexed vector as a smoke test. The second applies a department filter, demonstrating the metadata constraint used by later retrievers.

In [ ]:
query_vector = embedding_records[0]["embedding"]
results = vector_store.query_index(result.index_path, query_vector, top_k=3)
assert results[0]["chunk_id"] == embedding_records[0]["chunk_id"]
print(json.dumps([{"chunk_id": x["chunk_id"], "score": x["score"], "department": x["department"]} for x in results], indent=2))
filtered = vector_store.query_index(result.index_path, query_vector, top_k=3, department="Emergency Department")
assert all(x["department"] == "Emergency Department" for x in filtered)
print(f"Filtered Emergency Department results: {len(filtered)}")

## 4. Inspect report and outputs

The manifest records the backend contract. The report adds file size and smoke-query evidence, while the audit CSV provides one traceability row per vector.

In [ ]:
report = json.loads(result.report_path.read_text(encoding="utf-8"))
failed = json.loads(result.failed_path.read_text(encoding="utf-8"))
assert report["vector_count"] == 102
assert failed == []
print(json.dumps(report, indent=2))

## 5. Display index-composition plots

These plots confirm that all expected operational departments and source types are represented in the persisted index.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(result.department_plot_path)))
display(Image(filename=str(result.source_plot_path)))

## Notebook and Python module differences

- The notebook explains the contract, previews validation, runs sample queries, displays plots, and supports human review.
- The Python module owns schema creation, float32 serialization, atomic replacement, integrity checks, metadata filtering, exact cosine search, reporting, plotting, and CLI behavior.
- The notebook imports the module instead of duplicating vector-store logic.

## Safety and next step

- Only approved or synthetic hospital text should be indexed.
- The index preserves source citations and safety labels; it does not generate clinical advice.
- Embedding model and dimension are stored with the index and must match query vectors.
- Continue with `07_retriever.ipynb` using `06_vector_index.sqlite3`.